# 📖 Bible Retrieval-Augmented Generation (RAG) System
### Built with SentenceTransformers, Manual NumPy Cosine Similarity, and Google Flan-T5

This notebook implements a complete, self-contained RAG system with **zero external retrieval/chunking frameworks** (no LangChain, no LlamaIndex, no FAISS). Everything is built transparently from scratch:

1. **Manual Text Chunking:** Pure Python sliding-window and verse-aware chunking.
2. **Vector Embeddings:** `sentence-transformers/all-MiniLM-L6-v2`.
3. **Retrieval Engine:** Custom **NumPy Cosine Similarity** top-$k$ search.
4. **Answer Generation:** `google/flan-t5-small` with automatic fallback:
   - Tries `pipeline('text2text-generation')`
   - Falls back directly to `AutoModelForSeq2SeqLM.generate` if pipeline is unavailable.

## 1. Install Required Packages
Run this cell in Google Colab to install `sentence-transformers`, `transformers`, `torch`, and `numpy`.

In [ ]:
!pip install -q sentence-transformers transformers torch numpy tqdm

## 2. Ingest the Bible Dataset
We provide automated download options (full King James Bible) and a built-in canonical sample dataset covering Genesis, Exodus, Psalms, Proverbs, Matthew, John, 1 Corinthians, Galatians, and Revelation.

In [ ]:
import urllib.request
import os

# Check if sample_bible.txt exists, if not write standard biblical text
bible_file = 'sample_bible.txt'

# If you want the full 66 books of the Bible, set DOWNLOAD_FULL_BIBLE = True
DOWNLOAD_FULL_BIBLE = False

def load_or_download_bible(download_full=False):
    if download_full:
        full_file = 'full_bible_kjv.txt'
        if os.path.exists(full_file):
            with open(full_file, 'r', encoding='utf-8') as f:
                return f.read()
        url = 'https://raw.githubusercontent.com/mxw/grmr/master/src/finalpos/data/alldata/kjv.txt'
        print(f'Downloading full Bible from {url}...')
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req) as resp:
                text = resp.read().decode('utf-8', errors='ignore')
                with open(full_file, 'w', encoding='utf-8') as f:
                    f.write(text)
                print(f'Downloaded {len(text):,} characters.')
                return text
        except Exception as e:
            print(f'Download failed ({e}), falling back to sample dataset.')

    with open('sample_bible.txt', 'r', encoding='utf-8') as f:
        return f.read()

raw_bible_text = load_or_download_bible(download_full=DOWNLOAD_FULL_BIBLE)
print(f'Loaded Bible text: {len(raw_bible_text):,} characters, {len(raw_bible_text.split()):,} words.')
print('Preview of first 300 characters:')
print(raw_bible_text[:300])

Loaded Bible text: 6,941 characters, 1,333 words.
Preview of first 300 characters:
THE BOOK OF GENESIS

Chapter 1
1:1 In the beginning God created the heaven and the earth.
1:2 And the earth was without form, and void; and darkness was upon the face of the deep. And the Spirit of God moved upon the face of the waters.
1:3 And God said, Let there be light: and there was light.
1:4 


## 3. Implement Manual Chunking (No External Frameworks)
Chunking is implemented with pure Python logic:
- **Word-level sliding window:** Splits text by words with a configurable chunk size and overlap.
- **Verse-aware chunker:** Groups sequential verses while preserving book titles and chapter context.

In [ ]:
def manual_sliding_window_chunking(text: str, chunk_size: int = 120, overlap: int = 25) -> list[str]:
    """
    Manually chunks text using a sliding window over words with overlap.
    """
    words = text.split()
    if not words:
        return []

    chunks = []
    step = max(1, chunk_size - overlap)
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size]).strip()
        if chunk:
            chunks.append(chunk)
        if i + chunk_size >= len(words):
            break
    return chunks

def manual_verse_chunking(text: str, verses_per_chunk: int = 4, overlap: int = 1) -> list[str]:
    """
    Groups consecutive Bible verses with metadata context.
    """
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    verse_items = []
    current_heading = "Bible"

    for line in lines:
        if line.startswith(("THE BOOK", "THE GOSPEL", "THE EPISTLE", "Psalm", "THE REVELATION")):
            current_heading = line
            continue
        if line.startswith("Chapter"):
            continue
        verse_items.append(f"[{current_heading}] {line}")

    if not verse_items:
        return manual_sliding_window_chunking(text, chunk_size=120, overlap=25)

    chunks = []
    step = max(1, verses_per_chunk - overlap)
    for i in range(0, len(verse_items), step):
        c = "\n".join(verse_items[i:i + verses_per_chunk])
        if c:
            chunks.append(c)
        if i + verses_per_chunk >= len(verse_items):
            break
    return chunks

# Create chunks using verse-aware strategy
bible_chunks = manual_verse_chunking(raw_bible_text, verses_per_chunk=4, overlap=1)
print(f'Total chunks created: {len(bible_chunks)}')
print('\nSample Chunk #1:')
print(bible_chunks[0])

Total chunks created: 23

Sample Chunk #1:
[THE BOOK OF GENESIS] 1:1 In the beginning God created the heaven and the earth.
[THE BOOK OF GENESIS] 1:2 And the earth was without form, and void; and darkness was upon the face of the deep. And the Spirit of God moved upon the face of the waters.
[THE BOOK OF GENESIS] 1:3 And God said, Let there be light: and there was light.
[THE BOOK OF GENESIS] 1:4 And God saw the light, that it was good: and God divided the light from the darkness.


## 4. Vector Embeddings with `sentence-transformers/all-MiniLM-L6-v2`
We generate 384-dimensional dense semantic vectors for each Bible chunk.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embed_model_name = 'sentence-transformers/all-MiniLM-L6-v2'
print(f'Loading embedding model: {embed_model_name}...')
embedder = SentenceTransformer(embed_model_name)

# Encode all chunks
print(f'Encoding {len(bible_chunks)} chunks...')
chunk_embeddings = embedder.encode(bible_chunks, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)

print(f'Corpus embeddings matrix shape: {chunk_embeddings.shape}')
print(f'Data type: {chunk_embeddings.dtype}')

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding 23 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Corpus embeddings matrix shape: (23, 384)
Data type: float32


## 5. Manual NumPy Cosine Similarity & Top-$K$ Retrieval
Using pure NumPy linear algebra:
$$\text{Cosine Similarity}(q, d) = \frac{q \cdot d}{\|q\|_2 \|d\|_2}$$
Top-$K$ indices are extracted via `np.argsort`.

In [ ]:
def manual_cosine_similarity(query_vec: np.ndarray, doc_matrix: np.ndarray) -> np.ndarray:
    """
    Manual NumPy implementation of Cosine Similarity.
    """
    q = np.asarray(query_vec).flatten()
    docs = np.asarray(doc_matrix)

    dot_products = np.dot(docs, q)
    q_norm = np.linalg.norm(q)
    doc_norms = np.linalg.norm(docs, axis=1)

    denominator = (doc_norms * q_norm) + 1e-12
    similarities = dot_products / denominator
    return similarities

def manual_retrieve_top_k(query: str, top_k: int = 3):
    """
    Embeds query and ranks chunks using manual cosine similarity.
    """
    q_vec = embedder.encode(query, convert_to_numpy=True, normalize_embeddings=True)
    scores = manual_cosine_similarity(q_vec, chunk_embeddings)
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for rank, idx in enumerate(top_indices, 1):
        results.append({
            'rank': rank,
            'score': float(scores[idx]),
            'text': bible_chunks[idx]
        })
    return results

# Test retrieval on a query
test_query = 'Who built the ark?'
retrieved = manual_retrieve_top_k(test_query, top_k=2)
print(f'Retrieval test for: "{test_query}"')
for r in retrieved:
    print(f"\nRank {r['rank']} (Score: {r['score']:.4f}):\n{r['text']}")

Retrieval test for: "Who built the ark?"

Rank 1 (Score: 0.4415):
[THE BOOK OF GENESIS] 6:13 And God said unto Noah, The end of all flesh is come before me; for the earth is filled with violence through them; and, behold, I will destroy them with the earth.
[THE BOOK OF GENESIS] 6:14 Make thee an ark of gopher wood; rooms shalt thou make in the ark, and shalt pitch it within and without with pitch.
[THE BOOK OF GENESIS] 6:19 And of every living thing of all flesh, two of every sort shalt thou bring into the ark, to keep them alive with thee; they shall be male and female.
[THE BOOK OF EXODUS] 20:1 And God spake all these words, saying,

Rank 2 (Score: 0.3908):
[THE BOOK OF GENESIS] 6:5 And God saw that the wickedness of man was great in the earth, and that every imagination of the thoughts of his heart was only evil continually.
[THE BOOK OF GENESIS] 6:8 But Noah found grace in the eyes of the LORD.
[THE BOOK OF GENESIS] 6:9 Noah was a just man and perfect in his generations, and Noah 

## 6. Generator: `google/flan-t5-small` with Automatic Pipeline Fallback
As requested:
- Attempts `transformers.pipeline('text2text-generation')`
- If unavailable or fails, catches the exception and falls back to `AutoModelForSeq2SeqLM` and `AutoTokenizer` calling `.generate()` directly.

In [ ]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

generator_model_id = 'google/flan-t5-small'
pipe_generator = None
seq2seq_model = None
seq2seq_tokenizer = None

try:
    print(f'Attempting to load pipeline("text2text-generation", model="{generator_model_id}")...')
    pipe_generator = pipeline('text2text-generation', model=generator_model_id)
    print('SUCCESS: transformers.pipeline initialized.')
except Exception as e:
    print(f'Pipeline unavailable ({e}). Falling back to AutoModelForSeq2SeqLM...')
    seq2seq_tokenizer = AutoTokenizer.from_pretrained(generator_model_id)
    seq2seq_model = AutoModelForSeq2SeqLM.from_pretrained(generator_model_id)
    print('SUCCESS: AutoModelForSeq2SeqLM fallback initialized.')

def generate_answer(query: str, retrieved_passages: list[str]) -> str:
    global pipe_generator, seq2seq_model, seq2seq_tokenizer

    context = "\n\n".join([f"Passage {i+1}: {p}" for i, p in enumerate(retrieved_passages)])
    prompt = (
        f"Answer the following question based ONLY on the provided Bible context.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        f"Answer:"
    )

    # Try pipeline first
    if pipe_generator is not None:
        try:
            res = pipe_generator(prompt, max_length=120, min_length=4, do_sample=False)
            return res[0]['generated_text'].strip()
        except Exception as err:
            print(f'Pipeline execution failed ({err}), switching to AutoModel.generate directly...')

    # Direct AutoModel generation fallback
    if seq2seq_model is None:
        seq2seq_tokenizer = AutoTokenizer.from_pretrained(generator_model_id)
        seq2seq_model = AutoModelForSeq2SeqLM.from_pretrained(generator_model_id)

    inputs = seq2seq_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
    with torch.no_grad():
        tokens = seq2seq_model.generate(
            **inputs,
            max_length=120,
            min_length=4,
            num_beams=2,
            early_stopping=True
        )
    return seq2seq_tokenizer.decode(tokens[0], skip_special_tokens=True).strip()

Attempting to load pipeline("text2text-generation", model="google/flan-t5-small")...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

Pipeline unavailable ("Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"). Falling back to AutoModelForSeq2SeqLM...


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

SUCCESS: AutoModelForSeq2SeqLM fallback initialized.


## 7. End-to-End RAG Demonstration
Let's test the complete pipeline on canonical biblical questions!

In [ ]:
def ask_bible(question: str, top_k: int = 3):
    print('=' * 80)
    print(f'QUESTION: {question}')
    print('=' * 80)

    # Step 1: Retrieval via manual numpy cosine similarity
    retrieved_items = manual_retrieve_top_k(question, top_k=top_k)

    print(f'\n[Retrieved Contexts via NumPy Cosine Similarity (Top {top_k})]:')
    for item in retrieved_items:
        preview = item['text'].replace('\n', ' ')
        if len(preview) > 120:
            preview = preview[:120] + '...'
        print(f"  [Rank {item['rank']} | Cosine Sim: {item['score']:.4f}]: {preview}")

    # Step 2: Generation via Flan-T5
    passages = [item['text'] for item in retrieved_items]
    answer = generate_answer(question, passages)

    print(f'\n[Generated Answer (Flan-T5)]:')
    print(f'>>> {answer}\n')
    return answer

# Run evaluation queries
sample_questions = [
    'What did God create on the first day?',
    'Who built the ark of gopher wood?',
    'What are the fruits of the Spirit?',
    'What does the Bible say about love in 1 Corinthians?',
    'The LORD is my shepherd; what shall I not want?'
]

for q in sample_questions:
    ask_bible(q, top_k=2)

QUESTION: What did God create on the first day?

[Retrieved Contexts via NumPy Cosine Similarity (Top 2)]:
  [Rank 1 | Cosine Sim: 0.5570]: [THE BOOK OF GENESIS] 1:26 And God said, Let us make man in our image, after our likeness: and let them have dominion ov...
  [Rank 2 | Cosine Sim: 0.5272]: [THE BOOK OF GENESIS] 1:4 And God saw the light, that it was good: and God divided the light from the darkness. [THE BOO...

[Generated Answer (Flan-T5)]:
>>> Man in his own image

QUESTION: Who built the ark of gopher wood?

[Retrieved Contexts via NumPy Cosine Similarity (Top 2)]:
  [Rank 1 | Cosine Sim: 0.4252]: [THE BOOK OF GENESIS] 6:13 And God said unto Noah, The end of all flesh is come before me; for the earth is filled with ...
  [Rank 2 | Cosine Sim: 0.3361]: [THE BOOK OF GENESIS] 6:5 And God saw that the wickedness of man was great in the earth, and that every imagination of t...

[Generated Answer (Flan-T5)]:
>>> Noah and Noah

QUESTION: What are the fruits of the Spirit?

[Retrieve

## 8. Interactive Query Cell
Type your own question below and run the cell to query the Bible RAG system:

In [ ]:
user_question = "Who is God?"
ask_bible(user_question, top_k=3)

QUESTION: Who is God?

[Retrieved Contexts via NumPy Cosine Similarity (Top 3)]:
  [Rank 1 | Cosine Sim: 0.3637]: [THE BOOK OF EXODUS] 20:1 And God spake all these words, saying, [THE BOOK OF EXODUS] 20:2 I am the LORD thy God, which ...
  [Rank 2 | Cosine Sim: 0.3411]: [THE REVELATION OF SAINT JOHN THE DIVINE] 21:1 And I saw a new heaven and a new earth: for the first heaven and the firs...
  [Rank 3 | Cosine Sim: 0.3189]: [THE GOSPEL ACCORDING TO SAINT JOHN] 1:1 In the beginning was the Word, and the Word was with God, and the Word was God....

[Generated Answer (Flan-T5)]:
>>> the LORD thy God



'the LORD thy God'

In [ ]:
import gradio as gr

def gradio_bible_rag(question: str, top_k: int):
    if not question.strip():
        return "Please enter a question.", ""

    # 1. Retrieval via manual NumPy cosine similarity
    retrieved_items = manual_retrieve_top_k(question, top_k=int(top_k))

    # 2. Generation via Flan-T5
    passages = [item['text'] for item in retrieved_items]
    answer = generate_answer(question, passages)

    # 3. Format retrieved scripture context with scores
    retrieval_display = ""
    for item in retrieved_items:
        retrieval_display += (
            f"### 📌 Rank {item['rank']} | Cosine Similarity: `{item['score']:.4f}`\n"
            f"```text\n{item['text']}\n```\n\n"
        )

    return answer, retrieval_display

# Build Gradio UI
with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue")) as demo:
    gr.Markdown("# 📖 Bible Retrieval-Augmented Generation (RAG) System")
    gr.Markdown(
        "Ask any question about the Bible. The system retrieves relevant scriptures using "
        "**manual NumPy cosine similarity** over `all-MiniLM-L6-v2` embeddings and generates answers with `google/flan-t5-small`."
    )

    with gr.Row():
        with gr.Column(scale=2):
            query_input = gr.Textbox(
                label="Enter your Bible question:",
                placeholder="e.g., What did God create on the first day?",
                lines=2
            )
            top_k_slider = gr.Slider(
                minimum=1, maximum=5, value=2, step=1,
                label="Top-K Scripture Passages to Retrieve"
            )
            submit_btn = gr.Button("🔍 Search & Generate Answer", variant="primary")

            gr.Examples(
                examples=[
                    ["What did God create on the first day?", 2],
                    ["Who built the ark of gopher wood?", 2],
                    ["What are the fruits of the Spirit?", 2],
                    ["What does 1 Corinthians say about love?", 2],
                    ["The LORD is my shepherd; what shall I not want?", 2]
                ],
                inputs=[query_input, top_k_slider]
            )

        with gr.Column(scale=3):
            answer_output = gr.Textbox(label="💡 Generated Answer (Flan-T5)", lines=3)
            with gr.Accordion("📜 Retrieved Scripture Context & Cosine Similarity Scores", open=True):
                context_output = gr.Markdown()

    submit_btn.click(
        fn=gradio_bible_rag,
        inputs=[query_input, top_k_slider],
        outputs=[answer_output, context_output]
    )

# Launch the web app (share=True generates a public link on Colab)
demo.launch(share=True, debug=False)

/tmp/ipykernel_2846/1309955873.py:25: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue")) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://976ad88922d236918e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
